# OpenAaaS 快速入门 / OpenAaaS Quick Start

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Wolido/OpenAaaS/main?filepath=examples%2Fquickstart.ipynb)

在浏览器中直接体验 OpenAaaS Python SDK，无需本地安装。
Run OpenAaaS Python SDK directly in your browser, no local installation needed.

## 简介 / Introduction

**OpenAaaS**（Open Agent-as-a-Service）是一个面向科学研究的智能体编排平台。
**OpenAaaS** (Open Agent-as-a-Service) is an agent orchestration platform for scientific research.

本 Notebook 将带你完成一次完整的科研任务流程：注册账号 → 发现服务 → 提交任务 → 获取并展示结果。
This notebook walks you through a complete research workflow: register → discover services → submit a task → retrieve and display results.

---

每一步都配有代码示例，点击上方的 **Binder** 徽章即可在浏览器中直接运行。
Each step includes runnable code examples. Click the **Binder** badge above to run everything in your browser.

In [ ]:
# 安装 pyopenaaas
# Install pyopenaaas
!pip install pyopenaaas -q

In [ ]:
import pyopenaaas

# 创建客户端，默认连接公共服务器 https://api.open-aaas.com
# Create client, default public server
client = pyopenaaas.Client()

In [ ]:
# 注册获取 API Key（无需预先准备，自动生成随机用户名）
# Register to get an API Key (auto-generated random username)
result = client.register()
print("API Key:", result["api_key"])

In [ ]:
# 查看公共服务器上有哪些科研服务
# Discover available scientific services
services = client.list_services()
for svc in services:
    print(f"- {svc.name} (ID: {svc.id})")

In [ ]:
# 获取第一个服务的详细用法说明
# Get detailed usage for the first service
usage = client.get_service_usage(services[0].id)
print(usage.usage)

In [ ]:
# 提交一个科研任务
# Submit a research task
task = client.submit_task(
    service_id=services[0].id,
    task_prompt="为我调研一下如何设计具有高温塑性且抗氧化的高熵合金",
    output_prompt="",
)
print(f"Task ID: {task.id}, Status: {task.status}")

In [ ]:
# 自动轮询等待任务完成（每10秒检查一次）
# Poll until task completes (check every 10 seconds)
task = client.wait_for_task(task.id, poll_interval=10.0)
print(f"Final status: {task.status}")

In [ ]:
# 下载所有结果文件到本地 .OpenAaaS/downloads/<task_id>/
# Download all result files
if task.is_success():
    paths = client.download_all_files(task.id)
    for p in paths:
        print(f"Saved: {p}")
else:
    print(f"Task failed: {task.error_message}")

In [ ]:
from IPython.display import Markdown, display

# 找到下载目录中的 response.md 文件并直接渲染展示
# Find the downloaded response.md and render it directly
download_dir = pyopenaaas._utils._get_download_dir(task.id)
md_files = list(download_dir.glob("*.md"))

if md_files:
    for md_file in md_files:
        print(f"\n{'='*60}")
        print(f"📄 {md_file.name}")
        print(f"{'='*60}\n")
        display(Markdown(md_file.read_text(encoding="utf-8")))
else:
    print("No .md files found in download directory.")

## 总结与扩展 / Summary & Next Steps

恭喜！你已经完成了 OpenAaaS 的第一次科研任务。接下来你还可以尝试：
Congratulations! You've completed your first research task with OpenAaaS. Here are more things to explore:

- **探索更多服务** — 使用 `client.list_services()` 查看其他科研智能体。
  **Explore more services** — Use `client.list_services()` to discover other scientific agents.
- **上传本地文件** — 通过 `input_files` 参数将论文、数据等作为任务输入上传。
  **Upload local files** — Pass `input_files` to upload papers, data, etc. as task inputs.
- **使用异步 API** — 在大型工作流中尝试 `pyopenaaas.AsyncClient` 并发提交多个任务。
  **Use the async API** — Try `pyopenaaas.AsyncClient` to submit multiple tasks concurrently in large workflows.
- **查看任务输出** — 通过 `task.result.stdout` 查看智能体的执行日志。
  **Check task output** — Inspect `task.result.stdout` for the agent's execution logs.

---

📖 更多文档请参阅 SDK 源码与 API 文档。
📖 For more documentation, please refer to the SDK source code and API docs.